In [6]:
import sympy as sp
from IPython.display import display, Math

# Configuración visual
sp.init_printing(use_latex='mathjax')
t, w = sp.symbols('t w', real=True)
j_math = sp.I
j_sym = sp.Symbol('j')
u_sym = sp.Function('u')

def u(argumento):
    return sp.Heaviside(argumento)

def format_math(expr):
    # Truco para que se dibuje la 'u' y la 'j' bonitas en pantalla
    return sp.latex(expr.replace(sp.Heaviside, u_sym).subs(sp.I, j_sym))

def generador_corrimiento_tiempo(f_t_examen):
    print("==================================================")
    print(" ⏱️ DECONSTRUCTOR DE TEOREMA: CORRIMIENTO EN EL TIEMPO")
    print("==================================================\n")

    display(Math(rf"\text{{1. Función original del examen: }} f(t) = {format_math(f_t_examen)}"))

    # ALGORITMO DE DETECCIÓN: Buscamos cuánto se movió la función (t0)
    t0 = 0
    for arg in sp.preorder_traversal(f_t_examen):
        if isinstance(arg, sp.Heaviside):
            # Resolvemos lo que está adentro de la 'u' para encontrar el corrimiento
            sol = sp.solve(arg.args[0], t)
            if sol:
                t0 = sol[0]
            break

    if t0 == 0:
        print("\n--> No se detectó ningún corrimiento (t0 = 0). ¡Usa la transformada directa!")
        return

    print(f"\nPASO 1: Identificación del Desplazamiento")
    print(f"Al observar la función, identificamos la forma estándar f(t - t0).")
    print(f"Despejando el argumento, obtenemos que el corrimiento temporal es t0 = {t0}.")

    # Extrayendo la base (sustituimos t por t + t0 para limpiar la función)
    f_base = sp.simplify(f_t_examen.subs(t, t + t0))
    print(f"\nPASO 2: Extracción de la Función Base")
    print("Reconstruimos la función original sin el corrimiento:")
    display(Math(rf"f_{{base}}(t) = {format_math(f_base)}"))

    print(f"\nPASO 3: Transformada de la Función Base")
    print("Por definición o tablas, la transformada de esta base causal es:")
    try:
        # Calculamos la base usando integración sin condiciones
        res_int = sp.integrate(f_base * sp.exp(-j_math*w*t), (t, -sp.oo, sp.oo), conds='none')
        if isinstance(res_int, sp.Piecewise):
            res_int = res_int.args[0][0]
        
        # Truco de racionalización para que la 'jw' quede abajo positiva
        num, den = sp.fraction(sp.simplify(res_int))
        num_format = sp.simplify(num * j_math)
        den_format = sp.simplify(den * j_math)
        
        display(Math(rf"F_{{base}}(\omega) = \frac{{{format_math(num_format)}}}{{{format_math(den_format)}}}"))
    except Exception as e:
        print(f"Error al calcular la base: {e}")
        return

    print(f"\nPASO 4: Aplicación del Teorema")
    print("El Teorema de Corrimiento en el Tiempo dicta que un desplazamiento implica multiplicar por una exponencial en frecuencia:")
    display(Math(rf"\mathfrak{{F}}\{{f(t - t_0)\}} = F_{{base}}(\omega) \cdot e^{{-j\omega t_0}}"))
    
    # Resultado Final
    F_final_num = num_format * sp.exp(-j_math * w * t0)
    F_final_den = den_format
    
    print("\n==================================================")
    print(" RESULTADO FINAL F(w) (Copia todo esto en tu hoja):")
    display(Math(rf"F(\omega) = \frac{{{format_math(F_final_num)}}}{{{format_math(F_final_den)}}}"))
    print("==================================================")

# ==========================================
# 📝 CONFIGURACIÓN DEL EXAMEN
# ==========================================
# Mete tu problema con 'u' aquí:
mi_f_tiempo = u(t+4) * sp.exp(-(t+4))

generador_corrimiento_tiempo(mi_f_tiempo)

 ⏱️ DECONSTRUCTOR DE TEOREMA: CORRIMIENTO EN EL TIEMPO



<IPython.core.display.Math object>


PASO 1: Identificación del Desplazamiento
Al observar la función, identificamos la forma estándar f(t - t0).
Despejando el argumento, obtenemos que el corrimiento temporal es t0 = -4.

PASO 2: Extracción de la Función Base
Reconstruimos la función original sin el corrimiento:


<IPython.core.display.Math object>


PASO 3: Transformada de la Función Base
Por definición o tablas, la transformada de esta base causal es:


<IPython.core.display.Math object>


PASO 4: Aplicación del Teorema
El Teorema de Corrimiento en el Tiempo dicta que un desplazamiento implica multiplicar por una exponencial en frecuencia:


<IPython.core.display.Math object>


 RESULTADO FINAL F(w) (Copia todo esto en tu hoja):


<IPython.core.display.Math object>

## 🛡️ MANUAL DE SUPERVIVENCIA: La Función Escalón $u(t)$

La función escalón unitario $u(t)$ (o $\theta(t)$) es un "interruptor" matemático. Su único trabajo es decir dónde empieza o dónde termina una función. 

### 1. La Regla de Decisión (El Triage)
Cuando veas una $u(t)$ en el examen, hazte esta pregunta: **¿El desplazamiento es perfectamente simétrico en toda la ecuación?**

* ✅ **CASO IDEAL (Simétrico):** Ej. $u(t-4) e^{-(t-4)}$.
  * *Estrategia:* Usa el **Script 4 (Deconstructor de Tiempo)**. Te redactará el Teorema de Corrimiento en el Tiempo paso a paso.
* 🚨 **CASO MUTANTE (Asimétrico o Pulso):** Ej. $u(t-2) e^{-t}$ (exponentes distintos) o $u(t+2) - u(t-2)$ (varias $u$).
  * *Estrategia:* Aborta el teorema. Usa el **Script 1 (El Músculo / Definición)** con el Escudo Anti-Escalón.

---

### 2. Cómo usar el Script 1 para "Casos Mutantes"

Si el problema es asimétrico, el ingeniero está evaluando si sabes **cambiar los límites de una integral**, no si te sabes un teorema. 

**En tu Jupyter Notebook (Script 1):**
Solo debes escribir la función exactamente como viene, multiplicando todo. Deja que el script integre de $-\infty$ a $\infty$ con `conds='none'`.
```python
# Ejemplo de configuración para un caso asimétrico:
mi_funcion_examen = u(t-2) * sp.exp(-t)
fourier_con_escalon_examen(mi_funcion_examen)
```

---

## 📝 FASE 3: Redacción del Procedimiento Manual (Solo para Casos Mutantes)

Si usaste el **Script B**, debes escribir estos 4 pasos en tu hoja de examen para asegurar los 100 puntos, ya que el ingeniero requiere ver cómo cambian los límites:

**Paso 1: Planteamiento General**
Escribe la integral de Fourier copiando la función del examen.
$$F(\omega) = \int_{-\infty}^{\infty} \left[ u(t-2) e^{-t} \right] e^{-j\omega t} dt$$

**Paso 2: La Destrucción de la $u$**
Justifica: *"La función $u(t-2)$ solo existe para $t > 2$."* Reescribe la integral con el nuevo límite inferior y elimina la letra $u$.
$$F(\omega) = \int_{2}^{\infty} e^{-t} e^{-j\omega t} dt$$

**Paso 3: Álgebra de Exponentes**
Agrupa las bases sumando sus exponentes.
$$F(\omega) = \int_{2}^{\infty} e^{-t(1+j\omega)} dt$$

**Paso 4: El Resultado del Script**
Justifica: *"Evaluando la antiderivada de 2 a infinito..."* y pega el resultado final que te dio el **Script B**.
$$F(\omega) = \frac{e^{-2(1+j\omega)}}{1+j\omega}$$